# 03 ? Farm A sequence preparation

This notebook reads the processed Farm A event files from `02_preprocessing.ipynb` and builds LSTM input windows. Each window keeps the original sensor readings in time order; it does not reduce them to averages or replace missing values.

The notebook does not train a model. It saves separate training, validation, and prediction windows so the next modeling step can use them.

## Window rules in simple language

- Each window contains **12 readings**. At ten minutes per reading, that is about **2 hours** of history.
- New windows start every **120 rows** (about 20 hours) to keep the output size manageable. These are initial settings that we can change later.
- A window stays inside one event, one time segment, and one data split. It cannot jump across a long gap.
- A window with any missing sensor reading is skipped. We do not fill blanks with zero.
- The model learns normal behavior from training rows whose `status_type_id` is 0, matching the project?s current trainer and the README?s definition of normal operation.
- The last 20% of each event?s supplied training rows are held for validation. This is a chronological holdout; it is never used as training input.
- Training windows containing a turbine/timestamp flagged as overlapping prediction data are skipped to reduce leakage.
- Prediction windows get an event-level target from metadata (`normal` = 0, `anomaly` = 1). This is an event label, not a hand label for each individual sensor row.

Risks to revisit: the 12-reading window and 120-row stride are initial settings; the event-level prediction label treats all prediction windows in an event alike; and the dataset?s README/metadata totals disagree overall.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

cwd=Path.cwd()
ROOT=next((p for p in (cwd,*cwd.parents) if (p/"data/processed/farm_a/events").is_dir()),None)
if ROOT is None: raise FileNotFoundError("Farm A processed event files are missing; run 02_preprocessing.ipynb first")
FARM_PROCESSED=ROOT/"data/processed/farm_a"
EVENT_DIR=FARM_PROCESSED/"events"
OUTPUT_DIR=FARM_PROCESSED/"features"
EVENT_SUMMARY=pd.read_csv(FARM_PROCESSED/"event_summary.csv")
SENSOR_SUMMARY=pd.read_csv(FARM_PROCESSED/"sensor_summary.csv")
FEATURES=SENSOR_SUMMARY["sensor_column"].tolist()
FILES=sorted(EVENT_DIR.glob("*.csv.gz"),key=lambda p:int(p.stem.split('.')[0]))
WINDOW_SIZE=12
STRIDE=120
NORMAL_STATUS={0}
VALIDATION_FRACTION=0.20
print("Farm A processed events:",len(FILES))
print("Sensor features:",len(FEATURES))
print(f"Window: {WINDOW_SIZE} readings (~{WINDOW_SIZE*10/60:g} hours); stride: {STRIDE} rows (~{STRIDE*10/60:g} hours)")
print("Train, validation, and prediction outputs will be kept separate.")

Farm A processed events: 22
Sensor features: 45
Window: 12 readings (~2 hours); stride: 120 rows (~20 hours)
Train, validation, and prediction outputs will be kept separate.


## 1. Check the input files

We verify that each processed event has the columns needed for windows and that the average sensor list is consistent. No raw files are read or changed here.

In [7]:
REQUIRED={"time_stamp","asset_id","id","train_test","status_type_id","event_id","event_label","segment_id","cross_event_split_overlap",*FEATURES}
if not FILES: raise FileNotFoundError(f"No .csv.gz event files found in {EVENT_DIR}")
for path in FILES:
    d=pd.read_csv(path,compression="gzip",nrows=0)
    missing=REQUIRED-set(d.columns)
    if missing: raise ValueError(f"{path.name} is missing columns: {sorted(missing)}")
print("All event files have the required columns.")
print("Labels in event summary:"); display(EVENT_SUMMARY.event_label.value_counts().to_frame("events"))
print("Example processed columns:"); display(pd.Series(pd.read_csv(FILES[0],compression="gzip",nrows=0).columns,name="column").to_frame().head(18))

All event files have the required columns.
Labels in event summary:


,events
event_label,
anomaly,12
normal,10


Example processed columns:


,column
0,time_stamp
1,asset_id
2,id
3,train_test
4,status_type_id
5,sensor_0_avg
6,sensor_1_avg
7,sensor_2_avg
8,wind_speed_4_avg
9,sensor_6_avg


## 2. Build windows

For model training, we use only normal operating status rows (`status_type_id` 0) from the supplied training split. The last 20% of each event?s supplied training rows become validation rows. Prediction windows come only from the supplied prediction split and retain their event-level label.

In [8]:
partitions={"train":[],"validation":[],"prediction":[]}
metadata={"train":[],"validation":[],"prediction":[]}
stats={k:{"candidates":0,"saved":0,"skipped_missing":0,"skipped_overlap":0,"too_short_runs":0} for k in partitions}

# Make one chronological train/validation boundary per turbine across ALL event files.
# The same turbine timestamp can occur in several event files, so a per-event split
# could put an identical timestamp in both training and validation.
train_timestamp_keys=set()
for path in FILES:
    split_rows=pd.read_csv(path,compression="gzip",usecols=["time_stamp","asset_id","train_test"])
    split_rows=split_rows.loc[split_rows.train_test.eq("train"),["time_stamp","asset_id"]].copy()
    split_rows["time_stamp"]=pd.to_datetime(split_rows.time_stamp,errors="coerce")
    if split_rows.time_stamp.isna().any(): raise ValueError(f"Invalid training timestamp found in {path.name}")
    train_timestamp_keys.update((str(asset),stamp) for asset,stamp in zip(split_rows.asset_id,split_rows.time_stamp))

timestamp_partition={}
keys_by_turbine={}
for turbine,stamp in train_timestamp_keys:
    keys_by_turbine.setdefault(turbine,[]).append(stamp)
for turbine,stamps in keys_by_turbine.items():
    stamps=sorted(set(stamps))
    if len(stamps)<2: raise ValueError(f"Not enough unique training timestamps for turbine {turbine}")
    cut=int(len(stamps)*(1-VALIDATION_FRACTION))
    cut=min(max(cut,1),len(stamps)-1)
    for stamp in stamps[:cut]: timestamp_partition[(turbine,stamp)]="train"
    for stamp in stamps[cut:]: timestamp_partition[(turbine,stamp)]="validation"

for path in FILES:
    d=pd.read_csv(path,compression="gzip")
    d["time_stamp"]=pd.to_datetime(d["time_stamp"],errors="coerce")
    if d.time_stamp.isna().any(): raise ValueError(f"Invalid timestamp found in processed {path.name}")
    d=d.sort_values(["time_stamp","source_row_order"],kind="stable").reset_index(drop=True)
    event_id=str(d.event_id.iloc[0]); event_label=str(d.event_label.iloc[0]).lower()
    if event_label not in {"normal","anomaly"}: raise ValueError(f"Unexpected event label {event_label!r} in {path.name}")
    # Supplied prediction rows stay held out; each supplied training timestamp uses the turbine-wide assignment above.
    d["model_partition"]="prediction"
    train_mask=d.train_test.eq("train")
    train_keys=[(str(asset),stamp) for asset,stamp in zip(d.loc[train_mask,"asset_id"],d.loc[train_mask,"time_stamp"])]
    missing_keys=[key for key in train_keys if key not in timestamp_partition]
    if missing_keys: raise ValueError(f"Training timestamps missing a global partition in event {event_id}")
    d.loc[train_mask,"model_partition"]=[timestamp_partition[key] for key in train_keys]
    # Group boundaries already prevent crossing event, split, and long time gaps.
    groups=d.groupby(["train_test","segment_id","model_partition"],sort=False,dropna=False)
    for (supplied_split,segment_id,partition),g in groups:
        if partition not in partitions: continue
        # Training/validation use normal operating status and exclude rows whose timestamps overlap prediction data elsewhere.
        if partition in {"train","validation"}:
            eligible=g.status_type_id.isin(NORMAL_STATUS) & ~g.cross_event_split_overlap.astype(bool)
            overlap_bad=g.cross_event_split_overlap.astype(bool)
            stats[partition]["skipped_overlap"] += int(overlap_bad.sum()) if partition=="train" else 0
        else:
            eligible=pd.Series(True,index=g.index)
        local_positions=np.flatnonzero(eligible.to_numpy())
        if len(local_positions)==0: continue
        # If a row was filtered out, break the run so a window cannot jump over it.
        run_splits=np.flatnonzero(np.diff(local_positions)>1)+1
        runs=np.split(local_positions,run_splits)
        for run_pos in runs:
            run=g.iloc[run_pos]
            if len(run)<WINDOW_SIZE:
                stats[partition]["too_short_runs"]+=1
                continue
            for start in range(0,len(run)-WINDOW_SIZE+1,STRIDE):
                window=run.iloc[start:start+WINDOW_SIZE]
                stats[partition]["candidates"]+=1
                x=window[FEATURES].to_numpy(dtype=np.float32)
                if not np.isfinite(x).all():
                    stats[partition]["skipped_missing"]+=1
                    continue
                partitions[partition].append(x)
                idx=len(metadata[partition])
                pred_target=(1 if event_label=="anomaly" else 0) if partition=="prediction" else np.nan
                metadata[partition].append({"partition":partition,"window_index":idx,"event_id":event_id,
                    "turbine_id":str(window.asset_id.iloc[0]),"event_label":event_label,"target":pred_target,
                    "status_codes":";".join(map(str,sorted(window.status_type_id.dropna().unique()))),
                    "segment_id":int(segment_id),"start_time":window.time_stamp.iloc[0],"end_time":window.time_stamp.iloc[-1],
                    "contains_cross_event_overlap":bool(window.cross_event_split_overlap.astype(bool).any()),
                    "source_split":str(supplied_split)})
                stats[partition]["saved"]+=1

# Fixed dimensions make empty partitions easy to detect and diagnose.
arrays={name:(np.stack(values).astype(np.float32) if values else np.empty((0,WINDOW_SIZE,len(FEATURES)),dtype=np.float32)) for name,values in partitions.items()}
META={name:pd.DataFrame(rows) for name,rows in metadata.items()}
print("Window counts:")
display(pd.DataFrame(stats).T)
for name,x in arrays.items(): print(f"{name}: shape={x.shape}, dtype={x.dtype}")
if not len(arrays["train"]): raise ValueError("No training windows were produced")
if not len(arrays["validation"]): raise ValueError("No validation windows were produced")
if not len(arrays["prediction"]): raise ValueError("No prediction windows were produced")

Window counts:


,candidates,saved,skipped_missing,skipped_overlap,too_short_runs
train,7094,7094,0,72098,44
validation,1046,1046,0,0,4
prediction,441,441,0,0,8


train: shape=(7094, 12, 45), dtype=float32
validation: shape=(1046, 12, 45), dtype=float32
prediction: shape=(441, 12, 45), dtype=float32


## 3. Review the prediction labels and overlap flags

The target is assigned from each event?s metadata label. It is useful for an initial event-level metric, but it does not identify which individual rows inside an event are faulty. Overlap flags are kept in the prediction window metadata so we can report metrics with and without affected windows later.

In [9]:
prediction_meta=META["prediction"]
print("Prediction windows by event label:"); display(prediction_meta.event_label.value_counts().to_frame("windows"))
print("Prediction windows marked with cross-event overlap:",int(prediction_meta.contains_cross_event_overlap.sum()),"of",len(prediction_meta))
print("Prediction examples:"); display(prediction_meta.head())
print("Train/validation windows are status 0 only:")
for name in ["train","validation"]:
    codes=set(code for value in META[name].status_codes for code in str(value).split(";") if code)
    print(name,sorted(codes))

Prediction windows by event label:


,windows
event_label,
normal,223
anomaly,218


Prediction windows marked with cross-event overlap: 335 of 441
Prediction examples:


,partition,window_index,event_id,turbine_id,event_label,target,status_codes,segment_id,start_time,end_time,contains_cross_event_overlap,source_split
0,prediction,0,0,0,anomaly,1,0,17,2023-08-04 06:10:00,2023-08-04 08:00:00,True,prediction
1,prediction,1,0,0,anomaly,1,0,17,2023-08-05 02:10:00,2023-08-05 04:00:00,True,prediction
2,prediction,2,0,0,anomaly,1,0,17,2023-08-05 22:10:00,2023-08-06 00:00:00,True,prediction
3,prediction,3,0,0,anomaly,1,4,17,2023-08-06 18:10:00,2023-08-06 20:00:00,True,prediction
4,prediction,4,0,0,anomaly,1,4,18,2023-08-07 09:10:00,2023-08-07 11:00:00,True,prediction


Train/validation windows are status 0 only:
train ['0']
validation ['0']


## 4. Save LSTM windows

We save one compressed NumPy file with the three arrays and feature names. A CSV maps each window back to its event, turbine, time range, partition, label, and overlap flag. These are derived outputs under `data/processed/farm_a/features/`; raw and earlier processed event files remain unchanged.

In [10]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
window_path=OUTPUT_DIR/"farm_a_lstm_windows.npz"
metadata_path=OUTPUT_DIR/"window_metadata.csv"
y_prediction=np.array([int(v) for v in META["prediction"].target],dtype=np.int8)
np.savez_compressed(window_path,
    X_train=arrays["train"],X_validation=arrays["validation"],X_prediction=arrays["prediction"],
    y_prediction=y_prediction,feature_names=np.asarray(FEATURES,dtype="U"),
    window_size=np.array(WINDOW_SIZE),stride=np.array(STRIDE))
all_metadata=pd.concat([META["train"],META["validation"],META["prediction"]],ignore_index=True)
all_metadata.to_csv(metadata_path,index=False)
print("Saved:",window_path.relative_to(ROOT))
print("Saved:",metadata_path.relative_to(ROOT))
print("Train shape:",arrays["train"].shape,"| validation:",arrays["validation"].shape,"| prediction:",arrays["prediction"].shape)
print("Prediction target counts (0=normal, 1=anomaly):",pd.Series(y_prediction).value_counts().sort_index().to_dict())

Saved: data\processed\farm_a\features\farm_a_lstm_windows.npz
Saved: data\processed\farm_a\features\window_metadata.csv
Train shape: (7094, 12, 45) | validation: (1046, 12, 45) | prediction: (441, 12, 45)
Prediction target counts (0=normal, 1=anomaly): {0: 223, 1: 218}


## Summary and next step

The notebook creates time-ordered LSTM inputs from the processed Farm A data. It skips windows with missing readings, trains only on status-0 rows, holds out the last fifth of each event?s training rows for validation, and keeps the supplied prediction section separate. It also skips training windows touching timestamps that overlap prediction data in another event.

The next step is to review these window counts and then update the model-training notebook/code to load `farm_a_lstm_windows.npz`. The overlap flag and event-level target should be considered when reporting evaluation metrics.